# Retrieval-augmented generation with LLM4S in Jupyter

Embed a handful of documents, store them in the in-memory vector store, find the ones closest to a question and
answer from them. The notebook runs **offline** as committed: a small word-hashing embedder stands in for an
embedding model and `llm4s-testkit`'s `ScriptedLLMClient` for the chat model. The cell marked
**Switch to a real provider** uses real ones.

Setup (Almond, the kernel, JDK 21): [Notebooks guide](https://llm4s.org/guide/notebooks).

In [ ]:
//> using dep org.llm4s::llm4s-core:0.5.1
//> using dep org.llm4s::llm4s-rag:0.5.1
//> using dep org.llm4s::llm4s-testkit:0.5.1

## A scripted embedder

Each word is hashed into one of 64 dimensions, so texts that share words get similar vectors. It is deterministic
and needs no model; it is not a semantic embedding.

In [ ]:
import org.llm4s.llmconnect.config.EmbeddingModelConfig
import org.llm4s.llmconnect.model.{ EmbeddingRequest, EmbeddingResponse }
import org.llm4s.llmconnect.provider.EmbeddingProvider
import org.llm4s.types.Result

object WordHashEmbedder extends EmbeddingProvider {
  val dimensions = 64

  def vector(text: String): Seq[Double] = {
    val counts = Array.fill(dimensions)(0.0)
    text.toLowerCase.split("[^a-z0-9]+").filter(_.length > 2).foreach(word => counts(java.lang.Math.floorMod(word.hashCode, dimensions)) += 1)
    val norm = math.sqrt(counts.map(c => c * c).sum)
    counts.toSeq.map(c => if (norm == 0) 0.0 else c / norm)
  }

  def embed(request: EmbeddingRequest): Result[EmbeddingResponse] =
    Right(EmbeddingResponse(embeddings = request.input.map(vector), dim = Some(dimensions)))
}

## Switch to a real provider

Set `useRealProvider = true` to embed with the model `llm4s.embeddings` configures and answer with the chat provider
`llm4s.providers.provider` names. Configuration comes from system properties set in this cell before the first load:
`config.file` naming an `application.conf`, or single settings. Add the provider modules to the first cell and restart
the kernel. The `assert`s check only the scripted run.

In [ ]:
import org.llm4s.llmconnect.LLMClient
import org.llm4s.testkit.{ Reply, ScriptedLLMClient }

// SWITCH TO A REAL PROVIDER: set to true
val useRealProvider = false

// An application.conf with llm4s.providers and llm4s.embeddings sections:
// System.setProperty("config.file", "/path/to/application.conf")

val scripted = ScriptedLLMClient.sequence(
  Reply.text("Ship the order with the Fast tier: it arrives in one business day.")
)

val (embed, embeddingModel, client): (EmbeddingRequest => Result[EmbeddingResponse], EmbeddingModelConfig, LLMClient) =
  if (!useRealProvider) (WordHashEmbedder.embed, EmbeddingModelConfig("word-hash", WordHashEmbedder.dimensions), scripted)
  else {
    import org.llm4s.config.Llm4sConfig
    import org.llm4s.llmconnect.{ EmbeddingClient, LLMConnect }
    import org.llm4s.model.ModelRegistryService
    (for {
      registry <- Llm4sConfig.modelRegistryService()
      given ModelRegistryService = registry
      embeddings <- Llm4sConfig.embeddings()
      (provider, embeddingConfig) = embeddings
      embedder <- EmbeddingClient.from(provider, embeddingConfig)
      chat     <- Llm4sConfig.defaultProvider().flatMap(LLMConnect.getClient(_))
    } yield (embedder.embed, EmbeddingModelConfig(embeddingConfig.model, 0), chat))
      .fold(error => sys.error(error.formatted), identity)
  }

## Embed and store the documents

In [ ]:
import org.llm4s.vectorstore.{ VectorRecord, VectorStoreFactory }

val documents = Seq(
  "shipping-fast"     -> "Fast shipping delivers an order in one business day for an extra fee.",
  "shipping-standard" -> "Standard shipping delivers an order in five business days at no cost.",
  "returns"           -> "Returns are accepted within thirty days when the item is unused.",
  "warranty"          -> "Electronics carry a two year warranty covering manufacturing defects.",
  "payment"           -> "We accept credit cards, bank transfers and gift cards at checkout."
)

val store = VectorStoreFactory.inMemory().fold(error => sys.error(error.formatted), identity)

val stored = for {
  response <- embed(EmbeddingRequest(input = documents.map(_._2), model = embeddingModel))
  records = documents.zip(response.embeddings).map { case ((id, text), vector) =>
    VectorRecord(id, vector.map(_.toFloat).toArray, Some(text))
  }
  _     <- store.upsertBatch(records)
  count <- store.count()
} yield count

println(s"stored: $stored")
assert(stored == Right(documents.size.toLong))

## Search

In [ ]:
val question = "How fast can my order arrive with fast shipping?"

val found = for {
  response <- embed(EmbeddingRequest(input = Seq(question), model = embeddingModel))
  hits     <- store.search(response.embeddings.head.map(_.toFloat).toArray, topK = 2)
} yield hits

found.fold(error => println(error.formatted), _.foreach(hit => println(f"${hit.score}%.3f ${hit.record.id}: ${hit.record.content.getOrElse("")}")))
if (!useRealProvider) assert(found.map(_.head.record.id) == Right("shipping-fast"))

## Answer from the retrieved documents

In [ ]:
import org.llm4s.llmconnect.model.{ Conversation, SystemMessage, UserMessage }

val ragAnswer = for {
  hits <- found
  context = hits.flatMap(_.record.content).mkString("\n")
  completion <- client.complete(
    Conversation(Seq(
      SystemMessage(s"Answer only from these documents:\n$context"),
      UserMessage(question)
    ))
  )
} yield completion.content

ragAnswer.fold(error => println(error.formatted), println)
if (!useRealProvider) {
  assert(ragAnswer == Right("Ship the order with the Fast tier: it arrives in one business day."))
  assert(scripted.lastCall.exists(_.conversation.messages.head.content.contains("Fast shipping delivers")))
}
store.close()